Version 3 New features:
*   Removed pdf scraping
*   Code now attempts to download full text articles in XML
*   Uses PMC OA Web Service and/or FTP service
*   Tracks PMID that does not have corresponding PMCID


In [2]:
import requests
import xml.etree.ElementTree as ET
import time
import os
import logging
import gzip
import shutil
import pandas as pd
import json
import re
from datetime import datetime, timedelta
from google.colab import drive, userdata
from pathlib import Path
from urllib.parse import urlencode
from typing import Dict, List, Optional

In [3]:
# Variable declarations at start
BASE_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
RATE_LIMIT_DELAY = 0.34
DATABASE = "pubmed"
RETMODE_XML = "xml"
USE_HISTORY = "y"
DRIVE_BASE = "/content/drive/MyDrive/AI6129"
XML_PATH = f"{DRIVE_BASE}/xml"
LOG_PATH = f"{DRIVE_BASE}/logs"
GROUND_TRUTH_PATH = f"{DRIVE_BASE}/ground_truth"
TRACKING_FILE = f"{DRIVE_BASE}/download_tracker.csv"
MISSING_PMCID_LOG = f"{DRIVE_BASE}/missing_pmcids"
NCBI_API_KEY = userdata.get('ENTREZ_API')

In [4]:
# Configurations
PATHOGENS = {
    "hepatitis_a": {
        "name": "Hepatitis A",
        "mesh": "Hepatitis A",
        "term": "Hepatitis A"  # Single term, no synonyms
    },
    "hepatitis_e": {
        "name": "Hepatitis E",
        "mesh": "Hepatitis E",
        "term": "Hepatitis E"  # Single term, no synonyms
    }
}

# PMC specific URLs
PMC_OA_SERVICE = "https://www.ncbi.nlm.nih.gov/pmc/utils/oa/oa.fcgi"
PMC_PDF_BASE = "https://www.ncbi.nlm.nih.gov/pmc/articles"

# Enhanced rate limiting
PMC_RATE_LIMIT_DELAY = 1.0  # PMC requires slower rate
MAX_RETRIES = 3
BATCH_SIZE = 10

# File size limits (in MB)
MAX_XML_SIZE_MB = 10
COMPRESS_THRESHOLD_MB = 1

# Logging configuration
LOG_LEVEL = logging.INFO
LOG_FORMAT = '%(asctime)s - %(levelname)s - %(funcName)s - %(message)s'

In [5]:
#Logging Functions
def setup_logging():
  os.makedirs(LOG_PATH, exist_ok=True)
  os.makedirs(MISSING_PMCID_LOG, exist_ok=True)

  log_filename = f"{LOG_PATH}/pubmed_download_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

  logging.basicConfig(
      level=LOG_LEVEL,
      format=LOG_FORMAT,
      handlers=[logging.FileHandler(log_filename, encoding = 'utf-8'),
                logging.StreamHandler()],
      force=True,
  )

  logging.info(f"Logging initialized. Log file: {log_filename}")
  logging.info("PubMed Article Downloader v3")



def mount_google_drive():
  try:
    drive.mount('/content/drive', force_remount=False)
    os.makedirs(XML_PATH, exist_ok=True)
    os.makedirs(LOG_PATH, exist_ok=True)
    os.makedirs(MISSING_PMCID_LOG, exist_ok=True)
    logging.info("Google Drive mounted successfully")
    return True
  except Exception as e:
    logging.error(f"Failed to mount Google Drive: {str(e)}")
    return False

In [6]:
def build_pathogen_search_query(pathogen_key):
  """
    Build PubMed search query for single pathogen

    Args:
      pathogen_key: Key for pathogen in PATHOGENS dict

    Returns:
      str: Formatted PubMed search query
  """

  pathogen = PATHOGENS[pathogen_key]
  term = pathogen['term']
  mesh = pathogen['mesh']

  full_query = (f'(({term}[Title] OR {term}[Abstract] OR "{term}"[Methods - Key Terms]) '
                f'OR ({mesh}[MeSH Terms] OR {term}[All Fields]) '
                f'NOT ((review[All Fields] OR "review literature as topic"[MeSH Terms] OR review[All Fields]) '
                f'NOT Overview[All Fields]))')

  logging.info(f"Built search query for {pathogen['name']}")
  logging.info(f"Query: {full_query}")

  return full_query

In [7]:
#XML manipulation functions

def fetch_pmc_fulltext_xml(pmcid, retry_count=0):
  try:
    url = construct_pmc_fetch_url(pmcid)
    pmcid_clean = pmcid.replace('PMC', '', 1) if pmcid.startswith('PMC') else pmcid
    logging.info(f"Fetching full text XML for {pmcid_clean}")

    time.sleep(PMC_RATE_LIMIT_DELAY)
    response = requests.get(url, timeout=30)

    if response.status_code == 200:
      xml_content = response.text

      if '<body>' in xml_content or '<abstract>' in xml_content:
        logging.info(f"Successfully retrieved full-text XML for PMC{pmcid_clean}")
        return xml_content
      else:
        logging.warning(f"Response may contain metadata only for PMC{pmcid_clean}")
        return xml_content
    elif response.status_code == 404:
      logging.error(f"Full-text XML not found for PMC{pmcid_clean}")
      return None
    else:
      raise Exception(f"HTTP {response.status_code}")

  except Exception as e:
    logging.error(f"Failed to fetch full-text for PMC{pmcid_clean}: {str(e)}")

    if retry_count < MAX_RETRIES:
      logging.info(f". Retrying. Fetching full-text for PMC{pmcid_clean} (attempt {retry_count + 1})")
      time.sleep(RATE_LIMIT_DELAY * 2)
      return fetch_pmc_fulltext_xml(pmcid, retry_count + 1)

    return None

def save_xml_with_compression(xml_content, identifier, search_date):
  try:
    filename = f"{identifier}_{search_date}.xml"
    filepath = os.path.join(XML_PATH, filename)

    #Check if already exists
    if os.path.exists(filepath) or  os.path.exists(f"{filepath}.gz"):
      logging.info(f"XML for {identifier} already exists - skipping")
      return True

    #write XML
    with open(filepath, 'w', encoding='utf-8') as f:
      f.write(xml_content)

    #check file size
    file_size_mb = os.path.getsize(filepath) / (1024 * 1024)
    logging.info(f"Saved XML for {identifier}: {file_size_mb:.2f} MB")

    #compress if over threshold
    if file_size_mb > COMPRESS_THRESHOLD_MB:
      compressed_filepath = f"{filepath}.gz"
      with open(filepath, 'rb') as f_in:
        with gzip.open(compressed_filepath, 'wb') as f_out:
          shutil.copyfileobj(f_in, f_out)

      #remove original if compression successful
      os.remove(filepath)
      compressed_size_mb = os.path.getsize(compressed_filepath) / (1024 * 1024)
      logging.info(f"Compressed XML for {identifier}: {compressed_size_mb:.2f} MB")

    return True

  except Exception as e:
    logging.error(f"Failed to save XML for  {identifier}: {str(e)}")
    return False


In [8]:
#Extract functions
def extract_pmid_list(xml_response):
    root = ET.fromstring(xml_response)
    pmid_list = []
    id_list = root.find('IdList')

    if id_list is not None:
        for id_element in id_list.findall('Id'):
          pmid_list.append(id_element.text)
    return pmid_list

def extract_publication_date(article):

  pub_date = None
  data_element = article.find('.//PubDate')
  if data_element is not None:
    year = data_element.find('.//Year')
    month = data_element.find('.//Month')
    day = data_element.find('.//Day')

    if year is not None:
      pub_date = f"{year.text}"
      if month is not None:
        pub_date += f"-{month.text}"
        if day is not None:
          pub_date += f"-{day.text}"
  return pub_date

def extract_abstract_text(article):
  abstract = "Abstract not available"
  abstract_element = article.find('.//Abstract')
  if abstract_element is not None:
    abstract_texts = []
    for abstract_text in abstract_element.findall('AbstractText'):
      label = abstract_text.get('Label', '')
      text = abstract_text.text if abstract_text.text is not None else ''

      if label and text:
        abstract_texts.append(f'{label}: {text}')
      elif text:
        abstract_texts.append(text)

    if abstract_texts:
      abstract = ' '.join(abstract_texts)

  return abstract

def extract_total_count(xml_response):
  root = ET.fromstring(xml_response)
  count_element = root.find('Count')

  if count_element is not None and count_element.text is not None:
    return int(count_element.text)
  return 0

In [9]:
def convert_pmid_to_pmcid(pmid, retry_count=0):
  try:
    url = construct_elink_url(pmid)

    logging.debug(f"Converting PMID {pmid} to PMCID")
    time.sleep(RATE_LIMIT_DELAY)

    response = requests.get(url, timeout=30)
    if response.status_code != 200:
      logging.warning(f"elink returned HTTP {response.status_code} for PMID {pmid}")
      return None

    root = ET.fromstring(response.text)
    linksetdb = root.find('.//LinkSetDb')

    if linksetdb is None:
      logging.warning(f"No PMC link found for PMID {pmid}")
      return None

    link_id = linksetdb.find('.//Link/Id')
    if link_id is not None and link_id.text:
      pmcid_num = link_id.text
      pmcid = 'PMC' + pmcid_num
      logging.info(f"Converted PMID {pmid} to PMCID {pmcid}")
      return pmcid

    logging.warning(f"No PMCID found in response for PMID {pmid}")
    return None

  except ET.ParseError as e:
    logging.error(f"Failed to parse XML for PMID {pmid}: {str(e)}")
    return None

  except Exception as e:
    logging.error(f"Failed to convert PMID {pmid} to PMCID: {str(e)}")

    if retry_count < MAX_RETRIES:
      logging.info(f". Retrying. Converting PMID {pmid} (attempt {retry_count + 1})")
      time.sleep(RATE_LIMIT_DELAY * 2)
      return convert_pmid_to_pmcid(pmid, retry_count + 1)
  return None

def batch_convert_pmids_to_pmcids(pmid_list):
  pmid_to_pmcid_map = {}
  missing_pmcid_list = []

  logging.info(f"Starting PMID-to-PMCID conversion for {len(pmid_list)} articles")

  total_pmids = len(pmid_list)

  for i, pmid in enumerate(pmid_list, 1):
    pmcid = convert_pmid_to_pmcid(pmid)

    if pmcid is not None:
      pmid_to_pmcid_map[pmid] = pmcid
    else:
      missing_pmcid_list.append(pmid)
      logging.warning(f"PMID {pmid} has no PMC version")

  success_count = len(pmid_to_pmcid_map)
  success_rate = (success_count / total_pmids * 100) if total_pmids > 0 else 0

  logging.info(f"PMID-to-PMCID conversion completed. {success_count}/{total_pmids} successful ({success_rate:.1f}%)")
  logging.warning(f"Missing PMCIDs: {len(missing_pmcid_list)} articles")

  return pmid_to_pmcid_map, missing_pmcid_list

def track_missing_pmcid(tracking_df, pmid, reason):
  tracking_df = update_tracking_data(
      df=tracking_df,
      pmid=pmid,
      pmcid=None,
      pmcid_status='not_found',
      fulltext_xml_status='not_attempted',
      error_message=reason
  )

  missing_log_file = f"{MISSING_PMCID_LOG}/missing_pmcids_{datetime.now().strftime('%Y%m%d')}.csv"

  missing_record = {
      'pmid': pmid,
      'reason': reason,
      'date_checked': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
  }

  if os.path.exists(missing_log_file):
    missing_df = pd.read_csv(missing_log_file)
    missing_df = pd.concat([missing_df, pd.DataFrame([missing_record])], ignore_index=True)
  else:
    missing_df = pd.DataFrame([missing_record])

  missing_df.to_csv(missing_log_file, index=False)
  logging.warning(f"Tracked missing PMCID: PMID {pmid} - {reason}")

  return tracking_df

def generate_download_summary(tracking_df, date_start, date_end):
  summary_file = f"{LOG_PATH}/summary_{datetime.now().strftime('&Y%m%d')}.txt"

  current_date_str = datetime.now().strftime('%Y-%m-%d')
  current_run = tracking_df[tracking_df['download_date'].str.contains(current_date_str)]

  total_processed = len(current_run)
  pmcids_found = len(current_run[current_run['pmcid_status'] == 'found'])
  pmcids_not_found = len(current_run[current_run['pmcid_status'] == 'not_found'])
  pmcids_fetch_failed = len(current_run[current_run['pmcid_status'] == 'fetch_failed'])

  fulltext_success = len(current_run[current_run['fulltext_xml_status'] == 'success'])
  fulltext_failed = len(current_run[current_run['fulltext_xml_status'] == 'failed'])
  fulltext_not_attempted = len(current_run[current_run['fulltext_xml_status'] == 'not_attempted'])

  conversion_rate = (pmcids_found / total_processed * 100) if total_processed > 0 else 0
  success_rate = (fulltext_success / total_processed * 100) if total_processed > 0 else 0


  summary = f"""
    ===== PUBMED ARTICLE DOWNLOADER v3 - SUMMARY REPORT =====
    Date Range: {date_start} to {date_end}
    Execution Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

    PMID-TO-PMCID CONVERSION:
    -------------------------
    Total PMIDs Searched: {total_processed}
    PMCIDs Found: {pmcids_found} ({conversion_rate:.1f}%)
    PMCIDs Not Found: {pmcids_not_found}
    Conversion Fetch Failed: {pmcids_fetch_failed}

    FULL-TEXT XML RETRIEVAL:
    ------------------------
    Successful Downloads: {fulltext_success} ({success_rate:.1f}% of found)
    Failed Downloads: {fulltext_failed}
    Not Attempted: {fulltext_not_attempted}

    HISTORICAL DATA:
    ----------------
    Total Records in Database: {len(tracking_df)}
    Total Unique PMCIDs: {len(tracking_df[tracking_df['pmcid'].notna()]['pmcid'].unique())}

    FILES GENERATED:
    ----------------
    Tracking File: {TRACKING_FILE}
    Missing PMCIDs Log: {MISSING_PMCID_LOG}/missing_pmcids_{datetime.now().strftime('%Y%m%d')}.csv
    XML Files Directory: {XML_PATH}

    =========================================================
    """

  with open(summary_file, 'w') as f:
    f.write(summary)

  logging.info(f"Summary report saved to {summary_file}")
  print(summary)

  if pmcids_not_found > 0:
    missing_pmids = current_run[current_run['pmcid_status'] == 'not_found']['pmid'].tolist()
    missing_summary_file = f"{LOG_PATH}/missing_pmcids_summary_{datetime.now().strftime('%Y%m%d')}.txt"

    with open(missing_summary_file, 'w') as f:
      f.write(f"Articles Without PMC Versions ({len(missing_pmids)} total):\n")
      f.write("="*60 + "\n\n")
      for pmid in missing_pmids:
          f.write(f"PMID: {pmid}\n")
      f.write("\n" + "="*60 + "\n")
      f.write("Note: These articles are not available in PubMed Central.\n")
      f.write("They may be closed-access or not submitted to PMC.\n")

    logging.info(f"Missing PMCIDs summary saved to {missing_summary_file}")

In [10]:
def construct_elink_url(pmid):
  params ={
        'dbfrom': 'pubmed',
        'db': 'pmc',
        'linkname': 'pubmed_pmc',
        'id': pmid,
        'retmode': 'xml',
        'api_key': NCBI_API_KEY
  }
  return BASE_URL + 'elink.fcgi?' + urlencode(params)

def construct_pmc_fetch_url(pmcid):
  pmcid_clean = pmcid.replace('PMC','') if pmcid.startswith('PMC') else pmcid
  params ={
        'db': 'pmc',
        'id': pmcid_clean,
        'retmode': 'xml',
        'api_key': NCBI_API_KEY
  }
  return BASE_URL + 'efetch.fcgi?' + urlencode(params)

def construct_esearch_url(database, term, mindate, maxdate, datetype, retmax, usehistory):
    params = {
        'db': database,
        'term': term,
        'mindate': mindate,
        'maxdate': maxdate,
        'datetype': datetype,
        'retmax': retmax,
        'usehistory': usehistory,
        'retmode': 'xml',
        'api_key': NCBI_API_KEY
    }
    return BASE_URL + 'esearch.fcgi?' + urlencode(params)

def construct_efetch_url(database, pmids, retmode):
    id_string = ','.join(pmids)
    params = {
        'db': database,
        'id': id_string,
        'retmode': retmode,
        'api_key': NCBI_API_KEY
    }
    return BASE_URL + 'efetch.fcgi?' + urlencode(params)

def make_http_request(url, retry_count=0):
  try:
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return response.text
  except requests.exceptions.RequestException as e:
    logging.error(f"HTTP request failed: {str(e)}")
    if retry_count < MAX_RETRIES:
      logging.info(f". Retrying...(attempt {retry_count + 1})")
      time.sleep(RATE_LIMIT_DELAY * (retry_count + 1))
      return make_http_request(url, retry_count + 1)
    raise

def load_tracking_data():
  if os.path.exists(TRACKING_FILE):
    try:
      df = pd.read_csv(TRACKING_FILE)
      logging.info(f"Loaded {len(df)} existing records from tracking file")
      return df
    except Exception as e:
      logging.error(f"Failed to load tracking data: {str(e)}")

  #create new tracking dataframe
  df = pd.DataFrame(columns=['pmid', 'pmcid', 'pmcid_status','download_date', 'fulltext_xml_status', 'error_message'])
  return df

def update_tracking_data(df, pmid, pmcid, pmcid_status, fulltext_xml_status, error_message=None):
  new_record = {
      'pmid': pmid,
      'pmcid': pmcid,
      'pmcid_status': pmcid_status,
      'download_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
      'fulltext_xml_status': fulltext_xml_status,
      'error_message': error_message
  }

  if pmid in df['pmid'].values:
    idx = df[df['pmid'] == pmid].index[0]
    for key,value in new_record.items():
      df.at[idx, key] = value
    logging.debug(f"Updated existing record for PMID {pmid}")
  else:
    df = pd.concat([df, pd.DataFrame([new_record])], ignore_index=True)
    logging.debug(f"Added new record for PMID {pmid}")
  return df

def save_tracking_data(df):
  try:
    df.to_csv(TRACKING_FILE, index=False)
    logging.info(f"Saved tracking data with {len(df)} records")
  except Exception as e:
    logging.error(f"Failed to save tracking data: {str(e)}")

def get_previous_month_date_range():
  today = datetime.now()
  first_day_current_month = today.replace(day=1)
  last_day_previous_month = first_day_current_month - timedelta(days=1)
  first_day_previous_month = last_day_previous_month.replace(day=1)

  date_start = first_day_previous_month.strftime('%Y/%m/%d')
  date_end   = last_day_previous_month.strftime('%Y/%m/%d')

  logging.info(f"Date range for previous month: {date_start} to {date_end}")

  return date_start, date_end

def parse_month_input(month_input, year_input=None):
  """function expects a month and outputs the start of the month and end of the month date)"""

  month_names ={
      'January': '01', 'jan': '01',
      'February': '02', 'feb': '02',
      'March': '03', 'mar': '03',
      'April': '04', 'apr': '04',
      'May': '05', 'may': '05',
      'June': '06', 'jun': '06',
      'July': '07', 'jul': '07',
      'August': '08', 'aug': '08',
      'September': '09', 'sep': '09',
      'October': '10', 'oct': '10',
      'November': '11', 'nov': '11',
      'December': '12', 'dec': '12'
  }

  if year_input is None:
    year = datetime.now().year
  else:
    year = int(year_input)

  month_lower = month_input.lower().strip()

  if month_lower.isdigit():
    month = int(month_lower)
    if month < 1 or month > 12:
      raise ValueError(f"Invalid month number: {month_input}.  Must be between 1 and 12")
  else:
    month = int(month_names.get(month_lower))
    if month is None :
      raise ValueError(f"Invalid month:  {month_input}")

  if year < 1900 or year > datetime.now().year:
    raise ValueError(f"Invalid year: {year_input}")

  first_day = datetime(year, month, 1)

  if month == 12:
    last_day = datetime(year + 1, 1, 1) - timedelta(days=1)
  else:
    last_day = datetime(year, month + 1, 1) - timedelta(days=1)

  date_start = first_day.strftime('%Y/%m/%d')
  date_end = last_day.strftime('%Y/%m/%d')

  logging.info(f"Parsed {month_input} {year} to range: {date_start} - {date_end}")

  return date_start, date_end

In [11]:
def search_and_download_articles(pathogen_key,  date_start=None, date_end=None, interactive_mode=False):
  """
    Main function to search and download articles for single pathogen

    Args:

    Returns:
      Dataframe with tracking data
  """
  #Setup date range
  if date_start is None or date_end is None:
    date_start, date_end = get_previous_month_date_range()

  logging.info(f"Starting search for {PATHOGENS[pathogen_key]['name']}")
  logging.info(f"Date range: {date_start} to {date_end}")

  search_query = build_pathogen_search_query(pathogen_key)

  #Load tracking data
  tracking_df = load_tracking_data()

  #Perform search
  search_url = construct_esearch_url(
      database=DATABASE,
      term=search_query,
      mindate=date_start,
      maxdate=date_end,
      datetype='pdat',
      retmax='100',
      usehistory=USE_HISTORY
      )

  try:
    logging.info("Executing PubMed search...")
    response = make_http_request(search_url)
    time.sleep(RATE_LIMIT_DELAY)

    total_count = extract_total_count(response)
    pmid_list = extract_pmid_list(response)

    logging.info(f"Found {total_count} total articles, retrieved {len(pmid_list)} PMIDs")

    if interactive_mode:
      print(f"\n{'='*60}")
      print(f"SEARCH RESULTS")
      print(f"{'='*60}")
      print(f"Total articles matching criteria: {total_count}")
      print(f"Articles to be downloaded: {len(pmid_list)}")
      print(f"Date range: {date_start} to {date_end}")
      print(f"{'='*60}\n")

    if not pmid_list:
      logging.warning("No articles found for the specified criteria")
      return tracking_df

    logging.info("STEP 2: Converting PMIDs to PMCIDs...")
    pmid_to_pmcid_map, missing_pmcid_list = batch_convert_pmids_to_pmcids(pmid_list)

    for pmid in missing_pmcid_list:
      tracking_df = track_missing_pmcid(tracking_df, pmid, "No PMC version available")

    if interactive_mode:
      conversion_rate = len(pmid_to_pmcid_map) / len(pmid_list) * 100 if pmid_list else 0
      print(f"\nConversion Results:")
      print(f"  Success: {len(pmid_to_pmcid_map)}/{len(pmid_list)} ({conversion_rate:.1f}%)")
      print(f"  Missing: {len(missing_pmcid_list)}")
      print()

    logging.info("STEP 3: Fetching full-text XML from PMC...")

    #Process in batches
    success_count = 0
    failed_count = 0
    total_fetch = len(pmid_to_pmcid_map)

    for i, (pmid, pmcid) in enumerate(pmid_to_pmcid_map.items(), 1):

      if pmcid in tracking_df['pmcid'].values:
        existing_record = tracking_df[tracking_df['pmid'] == pmid].iloc[0]
        if existing_record['fulltext_xml_status'] == 'success':
          logging.info(f"Skipping {pmcid} - already processed")
          success_count += 1
          continue
      fulltext_xml = fetch_pmc_fulltext_xml(pmcid)

      if fulltext_xml:
        batch_date = datetime.now().strftime('%Y%m%d')
        xml_success = save_xml_with_compression(fulltext_xml, pmcid, batch_date)

        if xml_success:
          tracking_df = update_tracking_data(df=tracking_df,
                  pmid=pmid,
                  pmcid=pmcid,
                  pmcid_status = 'success',
                  fulltext_xml_status = 'success',
                  error_message = None
                  )
          success_count += 1
          logging.info(f"Successfully saved full-text for {pmcid}")
        else:
          tracking_df = update_tracking_data(
             df=tracking_df,
             pmid=pmid,
             pmcid=pmcid,
             pmcid_status='found',
             fulltext_xml_status='failed',
             error_message='XML save failed'
          )
          failed_count += 1
      else:
        tracking_df = update_tracking_data(
          df=tracking_df,
          pmid=pmid,
          pmcid=pmcid,
          pmcid_status='fetch_failed',
          fulltext_xml_status='failed',
          error_message='Full-text XML not available from PM'
       )
        failed_count += 1
        logging.error(f"Failed to save full-text for {pmcid}")

      time.sleep(RATE_LIMIT_DELAY)

      if i % 10 == 0:
        save_tracking_data(tracking_df)
        logging.info(f"Progress saved: {i}/{total_fetch} processed")

    save_tracking_data(tracking_df)

    #Log summary
    logging.info(f"Processing complete:")
    logging.info(f"  Full-text XML success: {success_count}")
    logging.info(f"  Full-text XML failed: {failed_count}")
    logging.info(f"  Missing PMCIDs: {len(missing_pmcid_list)}")

    #generate summary report
    generate_download_summary(tracking_df, date_start, date_end)
    return tracking_df

  except Exception as e:
    logging.error(f"Search and download failed: {str(e)}")
    save_tracking_data(tracking_df)
    raise

In [12]:
def extract_pmcid_from_filename(filename: str) -> Optional[str]:
  """
  Extract PMC ID from JSON filename.

  Expected patterns:
  - PMC1234567.json
  - PMC1234567_metadata.json
  - pmc1234567.json (case-insensitive)

  Returns:
      PMC ID string (e.g., "PMC1234567") or None if not found
  """
  # Pattern: PMC followed by digits
  pattern = r'(PMC\d+)'
  match = re.search(pattern, filename, re.IGNORECASE)

  if match:
      # Normalise to uppercase PMC
      return match.group(1).upper()

  return None

def collect_pmcids_from_directory(directory: str) -> List[str]:
  """
  Scan directory for JSON files and extract PMCIDs from filenames.
  Returns list of unique PMCIDs found.
  """

  pmcid_list = []
  files_scanned = 0
  files_with_pmcid = 0

  #get all files from ground_truth directory
  gt_files = list(Path(directory).glob("*.json"))

  print(f" directory {directory} has the following files: {gt_files}")

  for gt_file in gt_files:
      pmcid = extract_pmcid_from_filename(gt_file.name)
      files_scanned += 1

      if pmcid and pmcid not in pmcid_list:
          pmcid_list.append(pmcid)
          files_with_pmcid += 1
      else:
        logging.error(f"Could not extract PMCID from: {gt_file}")

  logging.info(f"Scanned {files_scanned} files, found {files_with_pmcid} unique PMCIDs")
  return pmcid_list


In [13]:
def run_interactive_mode():

  """Interactive mode for manual pathogen search"""

  logging.info("Entering interactive mode")

  print("\n=== PubMed Article Downloader - Interactive Mode ===\n")

  month_input = input("Enter month (e.g., 'June', 'Jun', or '6'): ")
  year_input = input("Enter year (press Enter for current year): ").strip()

  if not year_input:
    year_input = None

  try:
    date_start, date_end = parse_month_input(month_input, year_input)
  except ValueError as e:
    logging.error(f"Invalid input: {e}")
    print(f"Error: {e}")
    return

  print(f"Searching for publications from {date_start} to {date_end}")

  confirm = input("Proceed with download? (y/n): ").strip().lower()
  if confirm != 'y':
    print("Download canceled.")
    return

  if not mount_google_drive():
    raise Exception("Failed to mount Google Drive")

  pathogen_combinations = [
    ("hepatitis_a"),
    #("hepatitis_e", "hepatitis_a")

  ]

  for pathogen in pathogen_combinations:
    logging.info(f"Processing pathogen: {pathogen} ")

    try:
      tracking_df = search_and_download_articles(pathogen, date_start=date_start, date_end=date_end, interactive_mode=True)
      time.sleep(5)

    except Exception as e:
      logging.error(f"Failed to process {pathogen}: {str(e)}")
      continue

  print("\n✓ Interactive download complete")
  print(f"Check {LOG_PATH} for detailed logs and summaries")

def select_execution_mode():
  import sys

  if len(sys.argv) > 1:
    mode = sys.argv[1].lower()
    if mode == 'interactive':
      return 'interactive'
    elif mode == ['--scheduled', '-s']:
      return 'scheduled'

  print("\n=== PubMed Article Downloader ===")
  print("1. Interactive Mode (choose specific month)")
  print("2. Download ground-truth xml articles")
  print("3. Scheduled Mode (download previous month)")

  choice = input("\nSelect mode (1, 2 or 3): ").strip()

  if choice == '1':
      return 'interactive'
  elif choice == '2':
      return 'ground_truth'
  elif choice == '3':
      return 'scheduled'
  else:
      print("Invalid choice. Defaulting to scheduled mode.")
      return 'scheduled'


In [14]:
def download_ground_truth_xml() -> Dict:
  """
  Download full-text XML for each PMCID.
  Reuses existing v3 functions: fetch_pmc_fulltext_xml, save_xml_with_compression

  Returns summary dict with success/failure counts.
  """
  #check output directory exists
  OUTPUT_PATH = Path(XML_PATH) / "ground_truth"
  OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

  #Init counters
  success_count = 0
  skipped_count = 0
  failed_count  = 0
  failed_pmcids = []

  #Step 1: Get all PMCID for JSON filename
  logging.info(f"Step 1: Extracting PMCIDs from JSON filenames in directory {GROUND_TRUTH_PATH}")
  pmcid_list = collect_pmcids_from_directory(directory=GROUND_TRUTH_PATH)

  for idx, pmcid in enumerate(pmcid_list, 1):
    # Progress indicator every 10 items
    if idx % 10 == 0:
      logging.info(f"Progress: {idx}/{len(pmcid_list)}")

    #check if XML exists
    gt_filename_pattern = f"{pmcid}*.xml"
    first_match = next(OUTPUT_PATH.glob(gt_filename_pattern), None)

    if first_match:
      logging.info(f"Skipping {pmcid} - already processed")
      skipped_count += 1
      continue

    #Step 2: Fetch full-text XML
    xml_content = fetch_pmc_fulltext_xml(pmcid)

    if xml_content is not None:
      #Step 3: Save XML
      batch_date = datetime.now().strftime('%Y%m%d')
      save_success = save_xml_with_compression(xml_content, pmcid, batch_date)

      if save_success:
        success_count += 1
      else:
        failed_count += 1
        failed_pmcids.append(pmcid)
    else:
      failed_count += 1
      failed_pmcids.append(pmcid)
      logging.error(f"Failed to fetch XML for {pmcid}")


  summary = {
      'total'   : len(pmcid_list),
      'success' : success_count,
      'skipped' : skipped_count,
      'failed'  : failed_count,
  }

  return summary



In [15]:
def main():

  #initialize logging
  setup_logging()
  logging.info("Starting main execution")
  logging.info("v3 Changes: PMID-to-PMCID conversion, full-text XML from PMC, removed PDF scraping")
  try:

    mode = select_execution_mode()

    if mode == 'interactive':
      run_interactive_mode()

    elif mode == 'ground_truth':
      logging.info("Running in ground-truth mode")
      summary = download_ground_truth_xml()

      print("\n" + "="*60)
      print("DOWNLOAD SUMMARY")
      print("="*60)
      print(f"Total PMCIDs processed: {summary['total']}")
      print(f"Successfully downloaded: {summary['success']}")
      print(f"Skipped (already exist): {summary['skipped']}")
      print(f"Failed: {summary['failed']}")

      if summary['failed'] > 0:
        print(f"\nFailed PMCIDs: {summary['failed_pmcids']}")
    else: #scheduled mode
      #mount gdrive
      if not mount_google_drive():
        raise Exception("Failed to mount Google drive")

      #run for both pathogen combinations

      pathogen_search = ["hepatitis_a"]

      for pathogen in pathogen_search:
        logging.info(f"Processing pathogen: {pathogen}")

        try:
          tracking_df = search_and_download_articles(pathogen)

          #add delay
          time.sleep(3)

        except Exception as e:
          logging.error(f"Failed to process {pathogen}: {str(e)}")
          continue

    logging.info("All processing complete")
  except Exception as e:
    logging.error(f"Main execution failed: {str(e)}")
    raise

if __name__ == "__main__":
  main()



2025-11-25 14:06:05,394 - INFO - setup_logging - Logging initialized. Log file: /content/drive/MyDrive/AI6129/logs/pubmed_download_20251125_140605.log
2025-11-25 14:06:05,398 - INFO - setup_logging - PubMed Article Downloader v3
2025-11-25 14:06:05,401 - INFO - main - Starting main execution
2025-11-25 14:06:05,403 - INFO - main - v3 Changes: PMID-to-PMCID conversion, full-text XML from PMC, removed PDF scraping



=== PubMed Article Downloader ===
1. Interactive Mode (choose specific month)
2. Download ground-truth xml articles
3. Scheduled Mode (download previous month)

Select mode (1, 2 or 3): 2


2025-11-25 14:06:07,789 - INFO - main - Running in ground-truth mode
2025-11-25 14:06:07,795 - INFO - download_ground_truth_xml - Step 1: Extracting PMCIDs from JSON filenames in directory /content/drive/MyDrive/AI6129/ground_truth
2025-11-25 14:06:07,810 - INFO - collect_pmcids_from_directory - Scanned 352 files, found 352 unique PMCIDs
2025-11-25 14:06:07,814 - INFO - fetch_pmc_fulltext_xml - Fetching full text XML for 5388333


DEBUG directory repr: '/content/drive/MyDrive/AI6129/ground_truth'
DEBUG exists: True
DEBUG is_dir: True
 directory /content/drive/MyDrive/AI6129/ground_truth has the following files: [PosixPath('/content/drive/MyDrive/AI6129/ground_truth/PMC5388333.json'), PosixPath('/content/drive/MyDrive/AI6129/ground_truth/PMC9513250.json'), PosixPath('/content/drive/MyDrive/AI6129/ground_truth/PMC6449608.json'), PosixPath('/content/drive/MyDrive/AI6129/ground_truth/PMC8557873.json'), PosixPath('/content/drive/MyDrive/AI6129/ground_truth/PMC7168671.json'), PosixPath('/content/drive/MyDrive/AI6129/ground_truth/PMC8478212.json'), PosixPath('/content/drive/MyDrive/AI6129/ground_truth/PMC9206239.json'), PosixPath('/content/drive/MyDrive/AI6129/ground_truth/PMC9049261.json'), PosixPath('/content/drive/MyDrive/AI6129/ground_truth/PMC6307035.json'), PosixPath('/content/drive/MyDrive/AI6129/ground_truth/PMC6700665.json'), PosixPath('/content/drive/MyDrive/AI6129/ground_truth/PMC6016173.json'), PosixPath('/

2025-11-25 14:06:09,130 - INFO - fetch_pmc_fulltext_xml - Successfully retrieved full-text XML for PMC5388333
2025-11-25 14:06:09,142 - INFO - save_xml_with_compression - Saved XML for PMC5388333: 0.15 MB
2025-11-25 14:06:09,147 - INFO - fetch_pmc_fulltext_xml - Fetching full text XML for 9513250
2025-11-25 14:06:10,410 - INFO - fetch_pmc_fulltext_xml - Successfully retrieved full-text XML for PMC9513250
2025-11-25 14:06:10,422 - INFO - save_xml_with_compression - Saved XML for PMC9513250: 0.14 MB
2025-11-25 14:06:10,430 - INFO - fetch_pmc_fulltext_xml - Fetching full text XML for 6449608
2025-11-25 14:06:11,745 - INFO - fetch_pmc_fulltext_xml - Successfully retrieved full-text XML for PMC6449608
2025-11-25 14:06:11,759 - INFO - save_xml_with_compression - Saved XML for PMC6449608: 0.21 MB
2025-11-25 14:06:11,764 - INFO - fetch_pmc_fulltext_xml - Fetching full text XML for 8557873
2025-11-25 14:06:12,989 - INFO - fetch_pmc_fulltext_xml - Successfully retrieved full-text XML for PMC8557


DOWNLOAD SUMMARY
Total PMCIDs processed: 352
Successfully downloaded: 352
Skipped (already exist): 0
Failed: 0
